In [1]:
# General imports

import numpy as np
import pandas as pd
import geopandas as gpd
import netCDF4 as nc
import xarray as xr
import subprocess
import os
import sys
import math
import warnings
import re

# Plotting libraries

import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Machine learning

import torch
import torch.nn as nn
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.arima.model import ARIMA

# Data Processing - General SSP

In [2]:
# Get geography files
ssp_file = xr.open_dataset(os.getcwd() + "\\ssp2_population_total.nc4")

geo_csv = gpd.read_file("georef-canada-province-millesime.csv")
geo_shape = gpd.read_file("georef-canada-province-millesime.shp")

# Process shape ile into csv
geo_shape = geo_shape[geo_shape["year"] == "2021"]
df = pd.DataFrame(geo_shape.drop(columns=["geometry","year"]))

df.to_csv("Canada_shape_file.csv")

print("SSP Processing done. Execute the population aggregation R script in the next cell\n")



SSP Processing done. Execute the population aggregation R script in the next cell



# Execute R Script on Population Data

In [3]:
command = os.getcwd() + "\\population_agg_spatial.R"

# Use subprocess to execute the R script
process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
stdout, stderr = process.communicate()

# Print the output of R script
print("Output: ", stdout)
print("Errors: ", stderr)

Output:  b''
Errors:  b''


# Data Processing - StatsCAN

Source for data [here](https://www150.statcan.gc.ca/n1/pub/91-520-x/91-520-x2022001-eng.htm)

Download tables [here](https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=1710005701&pickMembers%5B0%5D=1.1&pickMembers%5B1%5D=3.1&pickMembers%5B2%5D=4.1&cubeTimeFrame.startYear=2021&cubeTimeFrame.endYear=2060&referencePeriods=20210101%2C20600101)

Population numbers are multiplied by 0.001


In [4]:
# Defining helper functions

def NaN_to_float(value):
    try:
        return float(value)
    except ValueError:
        return value

def is_float(number):
    try:
        float(number)
        return True
    except ValueError:
        return False
    
def interpolate_list(input_list): # Fix needed
    # Initialize the new list
    updated_list = []   
    # Loop through the input list
    for i in range(len(input_list) - 1):
        # Add new element to interpolation
        updated_list.append(input_list[i])
        # Calculate the interpolation between the current element and the next element
        interpolated_value = round((input_list[i] + input_list[i + 1])/2)
        # Append the interpolated value to the new list
        updated_list.append(interpolated_value)
    updated_list.append(input_list[-1])
    return updated_list

def remove_comma_separator(input_string):
    # This pattern matches numbers with commas as thousand separators
    pattern = r'(\d{1,3}(?:,\d{3})+)'
    
    # Function to replace the comma if the number is greater than 1000
    def replace_comma(match):
        number = int(match.group().replace(',', ''))
        return str(number) if number > 1000 else match.group()
    
    # Use re.sub() to apply the replace_comma function to all matches
    return re.sub(pattern, replace_comma, input_string)

def thousands_joiner(input_list):
    result_list = []
    iterator = 0
    while iterator < len(input_list):
        if input_list[iterator].isdigit() and int(input_list[iterator]):
            result_list.append(input_list[iterator] + input_list[iterator+1])
            iterator += 2
        else:
            result_list.append(input_list[iterator])
            iterator += 1
    return result_list

In [5]:
# Cleaning csv first

import csv

# Specify the path to your CSV file
scenario = input("StatsCan scenario: ")
input_file_path = os.getcwd() + f'\\StatsCan_Population_Predictions_{scenario}.csv'
output_file_path = os.getcwd() + '\\test.csv'

# Open the input CSV file in read mode and the output CSV file in write mode
with open(input_file_path, 'r') as input_file, open(output_file_path, 'w', newline='') as output_file:
    # Create a CSV reader and a CSV writer
    reader = csv.reader(input_file)
    writer = csv.writer(output_file)

    # Loop through each row in the input CSV file
    count_rows = 0
    for row in reader:
        # Remove all instances of " from each field in the row
        new_row = [field.replace('\"', '') for field in row]
        
        # Write the new row to the output CSV file
        if count_rows != 0:
            new_row = thousands_joiner(new_row)
            #new_row = [i for i in new_row if not (is_float(i) and float(i) < 1000)]
            
        writer.writerow(new_row)
        count_rows += 1


In [6]:
# Grab statscan data
population_df = pd.read_csv(os.getcwd() + "\\test.csv", sep=",",index_col="Geography", header=0)
population_df.dropna(inplace=True)

#print(population_df)

population_df["Total Population"] = population_df.sum(axis=1, skipna=True, numeric_only=True)
total_canada = population_df.sum(axis=0, skipna=True, numeric_only=True)
population_df = population_df.transpose()
population_df["Year Sum"] = total_canada
population_df = population_df.transpose()




#print(population_df)


In [7]:
population_df.to_csv(os.getcwd() + f"\\RESULT_SHEET_Canada_Population_Predictions_{scenario}.csv")

# Plots

First prepare dataframe for plots, then plot it using plotly

In [8]:

original_df = pd.read_csv(os.getcwd() + "\\Canada_Population.csv", sep=",", header=0, index_col=0)
input_gdp = pd.read_csv("Canada_GDP.csv", sep=",", header=0, index_col=0)

prov_list = [i for i in population_df.index]
prov_list = prov_list[:-1]


##########################################################

compare_dict = {"Year": [i for i in range(2020,2061,5)]} # Create year range


for prov in prov_list:
    statscan_list = list(population_df.loc[f"{prov}",population_df.columns == "2021"].values)
    for y in compare_dict["Year"]:
        if str(y) in population_df.columns:
            statscan_list.append(population_df.loc[f"{prov}", population_df.columns == str(y)].values[0])
        
    compare_dict[f"StatsCan_Projection_{prov}"] =  statscan_list

    compare_dict[f"SSP_Data_{prov}"] = list(original_df.loc[original_df["prov_name_e"] == f"['{prov}']",original_df.columns >= "sum.X2020"].values)
    compare_dict[f"{prov}_GDP"] = list(input_gdp.loc[input_gdp["prov_name_e"] == f"['{prov}']", input_gdp.columns >= "sum.X2020"].values)

    # Average 3 coordinates from the grid for each province
    a = list(np.round(np.mean(compare_dict[f"SSP_Data_{prov}"], axis=0)))
    a = a[0:5]
    #print(a)
    a = interpolate_list(a)
    #print(a)

    a2 = []
    for element in a:
        a2.append(element/10)
    compare_dict[f"SSP_Data_{prov}"] = a2

    gdp = list(np.round(np.mean(compare_dict[f"{prov}_GDP"], axis=0)))
    gdp = gdp[0:5]
    #print(gdp)
    gdp = interpolate_list(gdp)
    #print(gdp)
    compare_dict[f"{prov}_GDP"] = gdp

    b = []
    for k in compare_dict[f"StatsCan_Projection_{prov}"]: 
        k *= 1000 # Population numbers in the original sheet were scaled down by 1e3
        b.append(k)
    compare_dict[f"StatsCan_Projection_{prov}"] = b

#print(compare_dict)

#print(compare_dict["Year"],"\n")

for k,v in compare_dict.items():
    print(f"{k} ' : ' {len(v)}")



Year ' : ' 9
StatsCan_Projection_Newfoundland and Labrador ' : ' 9
SSP_Data_Newfoundland and Labrador ' : ' 9
Newfoundland and Labrador_GDP ' : ' 9
StatsCan_Projection_Prince Edward Island ' : ' 9
SSP_Data_Prince Edward Island ' : ' 9
Prince Edward Island_GDP ' : ' 9
StatsCan_Projection_Nova Scotia ' : ' 9
SSP_Data_Nova Scotia ' : ' 9
Nova Scotia_GDP ' : ' 9
StatsCan_Projection_New Brunswick ' : ' 9
SSP_Data_New Brunswick ' : ' 9
New Brunswick_GDP ' : ' 9
StatsCan_Projection_Quebec ' : ' 9
SSP_Data_Quebec ' : ' 9
Quebec_GDP ' : ' 9
StatsCan_Projection_Ontario ' : ' 9
SSP_Data_Ontario ' : ' 9
Ontario_GDP ' : ' 9
StatsCan_Projection_Manitoba ' : ' 9
SSP_Data_Manitoba ' : ' 9
Manitoba_GDP ' : ' 9
StatsCan_Projection_Saskatchewan ' : ' 9
SSP_Data_Saskatchewan ' : ' 9
Saskatchewan_GDP ' : ' 9
StatsCan_Projection_Alberta ' : ' 9
SSP_Data_Alberta ' : ' 9
Alberta_GDP ' : ' 9
StatsCan_Projection_British Columbia ' : ' 9
SSP_Data_British Columbia ' : ' 9
British Columbia_GDP ' : ' 9
StatsCan_Pro

In [9]:
compare_df = pd.DataFrame.from_dict(compare_dict)
#print(compare_df.head())

## Plotting Population

In [10]:


compare_fig = make_subplots(rows=len(prov_list) + 1, cols=1,
    subplot_titles=prov_list)


for i, prov in enumerate(prov_list):
    compare_fig.add_trace(go.Scatter(x=compare_df["Year"], 
        y=compare_df[f"StatsCan_Projection_{prov}"], name=f"StatsCan Prediction for {prov}"), row=i+1, col=1)
    
    compare_fig.append_trace(go.Scatter(x=compare_df["Year"],
        y=compare_df[f"SSP_Data_{prov}"], name=f"SSP_Data for {prov}"),i+1,1)
    
compare_fig.add_trace(go.Scatter(x=compare_df["Year"], y=population_df.loc["Year Sum",:],
    name="Total Population Prediction"), row=int(len(prov_list) + 1), col=1)

#compare_fig.add_trace(go.Scatter(x=compare_df["Year"], y=original_df))

for i in range(len(prov_list)):
    compare_fig.update_xaxes(title_text="Year", row=i+1, col=1)

compare_fig.update_layout(height=300*len(prov_list), width=700,title_text="Predictions for Each Province")

compare_fig.show()

# GDP Analysis Plots

Here we will analyse the GDP data for Canadian provinces

In [11]:
gdp_fig = px.line(compare_df, x="Year", y=f"{prov_list[5]}_GDP", 
    title="SSP GDP Projections")

ssp_gdp_df = pd.DataFrame()
ssp_gdp_df["Year"] = compare_df["Year"]

for i, prov in enumerate(prov_list):
    gdp_fig.add_scatter(x=compare_df["Year"], y=compare_df[f"{prov}_GDP"], name=prov)
    ssp_gdp_df[f"{prov}_GDP"] = compare_df[f"{prov}_GDP"]


gdp_fig.update_layout(xaxis_title="Year", yaxis_title="GDP [CAD$]")
gdp_fig.show()

ssp_gdp_df.to_csv(f"SSP_GDP_Projections.csv")




# Machine Learning - Predicting the Population Growth past 2043

Here we will employ machine learning to predict the population growth for all provinces past 2043, since the individual province data from StatsCan stops there. We can use the Canadian population growth predictions from StatsCan to validate the results afterwards

In [12]:
# Build input dataset
warnings.filterwarnings("ignore")

input_dataset = {"Year": population_df.columns}

for prov in prov_list:
    input_dataset[f"Population_{prov}"] = population_df.loc[f"{prov}",:].replace('..','NaN',regex=True).astype(float)

input_dataset = pd.DataFrame.from_dict(input_dataset)
input_dataset.drop_duplicates(keep="last", inplace=True)
input_dataset.dropna(inplace=True)

#print(input_dataset["Year"][18])


In [13]:
# Building training and testing datasets
warnings.filterwarnings("ignore")

population_target = f"Population_{input('Type province name: ')}"
print("Training for ", population_target, "\n")

## Getting cutoff for training
cutoff_year, training_cutoff = int(input_dataset["Year"][0]) + round(0.8*len(range(2021,2044))) - 1, round(0.8*len(range(2021,2044)))

print("\nCutoff year: ", cutoff_year, "\nIndex equivalent: ", training_cutoff)

# Splitting datasets

training_set = input_dataset.iloc[:training_cutoff,:]
training_years = training_set["Year"]
del training_set["Year"]
training_set = training_set[f"{population_target}"]

#print(training_set)
######################################################################

testing_set = input_dataset.iloc[training_cutoff:-1,:]
testing_years = testing_set["Year"]
del testing_set["Year"]

testing_set = testing_set[f"{population_target}"]

#print(testing_years)


Training for  Population_Alberta 


Cutoff year:  2038 
Index equivalent:  18


## Linear Regression

Here we will use Pytorch to create the model using a linear regression approach

In [14]:
# Try training and testing with a single province first, then do the entire dataset

x_lr, y_lr = training_years, training_set

torch_x = torch.tensor(training_years.values.astype(np.int64()))
torch_y = torch.tensor(training_set)

# Torch model
lr_features = torch_x.shape

torch_lr_model = nn.Linear(lr_features[0], lr_features[0]) # (inputs, outputs)

lr_loss = nn.MSELoss() # Error metrics: using root-mean-square error loss
lr_loss_history = []

lr_optim = torch.optim.Adam(torch_lr_model.parameters(), lr=0.01) # Optimizer
lr_epochs = 1 * 10**4 # More epochs give a lower RMSE

for ep in range(lr_epochs):
    # Forward pass
    torch_y_tilda = torch_lr_model(torch_x.float()) # Prediction
    curr_loss = lr_loss(torch_y_tilda, torch_y.float()) # Compare prediction and training value
    lr_loss_history.append(curr_loss)
    # Backward pass
    curr_loss.backward()
    # Update input and weight vector
    lr_optim.step()
    lr_optim.zero_grad() # Skip unnecessary information

lr_predictions = torch_lr_model(torch_x.float()).detach().numpy()

torch_RMSE = mean_squared_error(y_lr, lr_predictions, squared=False)
print(f"\nRMSE for {population_target}: {torch_RMSE}")




RMSE for Population_Alberta: 0.0001400118045188959


In [15]:
# Visuals

torch_figure = go.Figure()
torch_figure.add_trace(go.Scatter(x=training_years, y=lr_predictions,
    mode="lines", name=f"Predictions for {population_target}"))
torch_figure.add_trace(go.Scatter(x=training_years, y=y_lr,
    mode="lines", name="StatsCan Population Numbers"))
torch_figure.update_layout(title="PyTorch Linear Regression - Canadian Population Numbers",
    xaxis_title="Years", yaxis_title="Population Numbers [people x1,000]")


In [16]:
# Testing ML model
test_vector_years = [float(testing_years[-1])]
for i in range(int(testing_years[-1]),2060):
    test_vector_years.append(test_vector_years[-1] + 1)

testing_tensor = torch.tensor(test_vector_years)
test_predictions = torch_lr_model(testing_tensor[:len(torch_x)].float()).detach().numpy()

In [17]:
# View testing results

LR_test_figure = go.Figure()
LR_test_figure.add_trace(go.Scatter(x=test_vector_years, y=test_predictions, mode='lines', name='Predicted Population'))
LR_test_figure.add_trace(go.Scatter(x=test_vector_years, y=testing_set, mode='lines', name='StatsCan Predictions'))
LR_test_figure.update_layout(title=f'PyTorch Linear Regression - Test for {population_target}', xaxis_title='Years', yaxis_title='Population Numbers [people x1,000]')


# ARIMA Machine Learning

Using the ARIMA model, here we forecast the population growth for all provinces. The results are saved to a csv file

In [18]:
# Define ARIMA functions

# Function to fit ARIMA and predict future values
def predict_population(data):
    predictions = {}
    for index, col in data.iterrows():
        geography = index 
        time_series = pd.to_numeric(col[:], errors='coerce')
        time_series = time_series.dropna()
        model = ARIMA(time_series, order=(2, 1, 2))
        fitted_model = model.fit()
        forecast = fitted_model.forecast(steps=17)
        predictions[geography] = np.concatenate((time_series.values, forecast))
    return predictions

def plot_results(predictions, sc):
    Predict_figure = go.Figure()
    for geography, data in predictions.items():
        #if geography != 'Year Sum':
        Predict_figure.add_trace(go.Scatter(x=np.arange(2021, 2061), y=data, mode='lines',name=f"Population prediction for {geography}"))
    
    Predict_figure.update_layout(title=f'Population Forecast for {sc} Scenario',
        xaxis_title='Year', yaxis_title='Population (thousands)',
        width=800, height=500)
    Predict_figure.show()
    

def save_predicitions_to_csv(predictions,scenario):
    # Create a DataFrame from the predictions dictionary
    df = pd.DataFrame(predictions)
    # Set the year range as the index
    df.index = np.arange(2021, 2061)
    # Save to CSV
    df.to_csv(os.getcwd() + f'\\population_predictions_{scenario}.csv')
    print(f"Saved predictions to 'population_predictions_{scenario}.csv'")

def ARIMA_predict(data, sc):
    #data = load_data()
    predictions = predict_population(data)
    plot_results(predictions, sc)
    save_predicitions_to_csv(predictions, scenario=sc)  # Save the predictions to a CSV file

In [19]:
# Perform ARIMA predictions
ARIMA_df = population_df.iloc[:-1,:-1]
ARIMA_predict(ARIMA_df, scenario)

Saved predictions to 'population_predictions_MG1.csv'
